In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
import logging
from sklearn.feature_selection import VarianceThreshold

from sklearn.preprocessing import StandardScaler , MinMaxScaler, RobustScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
CLEANED_DATA_PATH="./../data/cleaned/cleaned_data.csv"
TRANSFORMATION_LOG_REPORT_PATH="./../reports/transformation_log_report.csv"
TRANSFORMATION_LOGGING_PATH="./../reports/transformation.log"

# Prepare for logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler(TRANSFORMATION_LOGGING_PATH),   # writes to file
        logging.StreamHandler(),               # prints to console
    ],
)

transformation_log_Report: list[dict] = []

In [ ]:
def log_transformation_action(stage: str, column: str, action: str, reason: str) -> None:
    transformation_log_Report.append({
        "stage": stage,
        "column": column,
        "action": action,
        "reason": reason,
    })
    logging.info(f"[LOG] {stage} | {column} | {action} | {reason}")

# load data

In [ ]:
df = pd.read_csv(CLEANED_DATA_PATH)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def show_histogram_with_stats(df, column_name):
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(5, 3))
    sns.histplot(df[column_name], kde=True, color="skyblue", bins=50, alpha=0.5)
    plt.axvline(df[column_name].mean(), color="red", linestyle="dashed", linewidth=1, label="Mean")
    plt.axvline(df[column_name].median(), color="green", linestyle="dashed", linewidth=1, label="Median")
    plt.legend()
    plt.title('Original ' + column_name.capitalize() + ' Distribution')
    plt.xlabel(column_name.capitalize())
    plt.ylabel('Count')
    plt.show()


# define our target 


In [ ]:
#split inot new col for the values of price_egp
show_histogram_with_stats(df, 'price_egp')


In [ ]:
# binning the price_egp using qcut to create 3 bins (0-100, 100-200, 200+)

df['price_egp_bin'] = pd.qcut(df['price_egp'], q=3, labels=[0,1,2])

In [ ]:
df['price_egp_bin'].value_counts()

# Split Data into Train and Test Sets   

In [ ]:
# split data into train and test sets
from sklearn.model_selection import train_test_split
X = df.drop(['price_egp', 'price_egp_bin'], axis=1)
# X = df.drop('price_category', axis=1)
y = df['price_egp_bin']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# split trian to train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:
# ### need to move 
# X_train.loc[X_train["amenities"] == "No amenities listed", "amenities"] = ""
# X_val.loc[X_val["amenities"] == "No amenities listed", "amenities"] = ""
# X_train["amenities_count"] = (
#     X_train["amenities"]
#     .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
# )

# X_val["amenities_count"] = (
#     X_val["amenities"]
#     .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
# )
# X_test.loc[X_test["amenities"] == "No amenities listed", "amenities"] = ""
# X_test["amenities_count"] = (   
#     X_test["amenities"]
#     .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
# )


In [ ]:
X_train_raw=X_train.copy()  
X_val_raw=X_val.copy()
X_test_raw=X_test.copy()


In [ ]:
X_train=X_train_raw
X_val=X_val_raw
X_test=X_test_raw

# Feature Scaling 
-   standardization
-   Min-Max Scaling
-   Robust Scaling 

**Transform numerical features to a common scale.**

### lat, lon 
lat:
- mean 29.88
- std 0.66
- min 25
- max 30.99
- median 30.01

lon:
- mean 31.46
- std 0.69
- min 27.97
- max 34.89
- median 31.25

**will use StandardScaler**

In [ ]:
show_histogram_with_stats(X_train, 'lat')
show_histogram_with_stats(X_train, 'lon')

In [ ]:
# apply scaling 
scaler = RobustScaler()
X_train['lat'] = scaler.fit_transform(X_train[['lat']])
X_val['lat'] = scaler.transform(X_val[['lat']])
X_test['lat'] = scaler.transform(X_test[['lat']])
show_histogram_with_stats(X_train, 'lat')
log_transformation_action(
    stage="Feature Scaling",
    column="lat",
    action="RobustScaler",
    reason="To handle outliers and scale the feature to a similar range as other features."
)



scaler = RobustScaler()
X_train['lon'] = scaler.fit_transform(X_train[['lon']])
X_val['lon'] = scaler.transform(X_val[['lon']])
X_test['lon'] = scaler.transform(X_test[['lon']])
show_histogram_with_stats(X_train, 'lon')

log_transformation_action(
    stage="Feature Scaling",
    column="lon",
    action="RobustScaler",
    reason="To handle outliers and scale the feature to a similar range as other features."     
)




###  (4) **area_value**
-   mean=145
-   median=145
-   Q1=116
-   Q3=173
-   max=765

**mean = median** <br>
**So normal distribution so will apply Standardization**

In [ ]:
show_histogram_with_stats(X_train, 'area_value')


In [ ]:

# apply scaling 
scaler = StandardScaler()
X_train['area_value'] = scaler.fit_transform(X_train[['area_value']])
X_val['area_value'] = scaler.transform(X_val[['area_value']])
X_test['area_value'] = scaler.transform(X_test[['area_value']])
#after
show_histogram_with_stats(X_train, 'area_value')
log_transformation_action(
    stage="Feature Scaling",
    column="area_value",
    action="StandardScaler",
    reason="To scale the feature to have mean 0 and variance 1, which can help some models perform better."
)


###  (6) **distance features**
- *dist_nearest_school_km*
- *dist_nearest_hospital_km*
- *dist_nearest_supermarket_km*
- *dist_nearest_mall_km*
- *dist_nearest_transit_station_km*
- *dist_nearest_cafe_restaurant_km*

right skewed <br>
**So will apply Robust Scaling**

In [ ]:
for col in X_train.columns:
    if col.startswith('dist_nearest'):
        show_histogram_with_stats(X_train, col)

In [ ]:
col=['dist_nearest_mall_km', 'dist_nearest_transit_station_km']

for c in col:
    scaler= StandardScaler()
    X_train[c] = scaler.fit_transform(X_train[[c]])
    X_val[c] = scaler.transform(X_val[[c]])
    X_test[c] = scaler.transform(X_test[[c]])
    show_histogram_with_stats(X_train, c)

    log_transformation_action(
        stage="Feature Scaling",
        column=c,
        action="StandardScaler",
        reason="To scale the feature to have mean 0 and variance 1, which can help some models perform better."     
    )




col =[ 'dist_nearest_school_km','dist_nearest_hospital_km', 'dist_nearest_supermarket_km','dist_nearest_cafe_restaurant_km']

for c in col:
    if c in X_train.columns:
        scaler = RobustScaler()
        X_train[c] = scaler.fit_transform(X_train[[c]])
        X_val[c] = scaler.transform(X_val[[c]])
        X_test[c] = scaler.transform(X_test[[c]])
        show_histogram_with_stats(X_train, c)
        log_transformation_action(
            stage="Feature Scaling",
            column=c,
            action="RobustScaler",
            reason="To handle outliers and scale the feature to a similar range as other features."     
        )

###  (7) **Count features**
- *school_count_within_3km*
- *hospital_count_within_3km*
- *supermarket_count_within_3km*
- *mall_count_within_3km*
- *transit_station_count_within_3km*
- *cafe_restaurant_count_within_3km*

bounded values non negative. <br>
**So will apply min-max scaling**

In [ ]:
for col in X_train.columns:
    if col.endswith('count_within_3km'):
        show_histogram_with_stats(X_train, col)

In [ ]:
for col in X_train.columns:
    if col.endswith('count_within_3km'):
        scaler = RobustScaler()
        X_train[col] = scaler.fit_transform(X_train[[col]])
        X_val[col] = scaler.transform(X_val[[col]])
        X_test[col] = scaler.transform(X_test[[col]])
        show_histogram_with_stats(X_train, col)
        log_transformation_action(
            stage="Feature Scaling",
            column=col,
            action="RobustScaler",
            reason="To handle outliers and scale the feature to a similar range as other features."     
        )

In [ ]:
X_train_Scaled=X_train.copy()  
X_val_Scaled=X_val.copy()
X_test_Scaled=X_test.copy()


In [ ]:
X_train=X_train_Scaled
X_val=X_val_Scaled
X_test=X_test_Scaled

# Feature Encoding 
-   One-Hot Encoding
-   Label Encoding
-   Target Encoding
-   Binary Encoding
-   Frequency Encoding
-   Rare Encoding

### (1) **Ordinal encode**  listing_level 
-   *standard* = 0
-  *featured* = 1
-  *premium* = 2
-  *hot* = 3
-  *superhot* = 4

as there is a clear order will apply Label Encoding

In [ ]:
listing_map = {
    "standard": 0,
    "featured": 1,
    "premium": 2,
    "hot": 3,
    "superhot": 4
}

for df_ in [X_train, X_val, X_test]:
    df_["listing_level"] = df_["listing_level"].map(listing_map)


log_transformation_action(
    stage="Feature Encoding",
    column="listing_level",
    action="Ordinal Encoding",
    reason="To convert the categorical feature into a numerical format, while preserving the ordinal relationship between the categories."     
)

### (2) **completion_status**              
-   *under-construction*    
-   *off_plan*      
-   *completed*     


### (3) **furnished**
-   *Unfurnished*    
-   *Unknown*         
-   *Furnished*        
-   *PARTLY*

we will apply one-hot encoding for both completion_status and furnished as they are nominal categorical variables with no clear order.

In [ ]:
cat_cols = ["completion_status", "furnished"]

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_val = pd.get_dummies(X_val, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)


X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

log_transformation_action(
    stage="Feature Encoding",
    column="completion_status, furnished",
    action="One-Hot Encoding",
    reason="To convert the categorical features into a numerical format, without assuming any ordinal relationship between the categories."     
)

### (4) Frequency encode 
-   *city*
-   *town*
-   *district*

In [ ]:
# can make with freq of city with high price
freq_cols = ["city", "town", "district"]

for col in freq_cols:
    freq_map = X_train[col].value_counts(normalize=True)

    for df in [X_train, X_val, X_test]:
        df[col] = df[col].map(freq_map).fillna(0)

log_transformation_action(
    stage="Feature Encoding",
    column="city, town, district",
    action="Frequency Encoding",
    reason="To convert the categorical features into a numerical format, while capturing the frequency of each category which may be related to the target variable."     
)

### (5) amenities convert to binary features


###  **convert bool to  int**


In [ ]:
for df in [X_train, X_val, X_test]:
    bool_cols = df.select_dtypes(bool).columns
    df[bool_cols] = df[bool_cols].astype(int)

log_transformation_action(
    stage="Feature Encoding",
    column=bool_cols.tolist(),
    action="Boolean to Integer Encoding",
    reason="To convert boolean features into a numerical format "     
)

In [ ]:
len(X_train.columns.tolist())

In [ ]:
X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()
X_test_encoded = X_test.copy()


In [ ]:
X_train=X_train_encoded
X_val=X_val_encoded
X_test=X_test_encoded

# Feature Interactions
-   Arithmetic Complications
-   Statistical Aggregations
-   Boolean and Logical Combinations


# arithmatic operations

In [ ]:
def add_arithmetic_features(df):
    df = df.copy()

    df["area_per_bedroom"] = df["area_value"] / (df["bedrooms"])
    df["area_per_bathroom"] = df["area_value"] / (df["bathroom"])
    df["bathroom_per_bedroom"] = df["bathroom"] / (df["bedrooms"] )
    df["total_rooms"] = df["bathroom"] + df["bedrooms"] 

    # Counts
    count_cols = [
        "school_count_within_3km",
        "hospital_count_within_3km",
        "supermarket_count_within_3km",
        "mall_count_within_3km",
        "transit_station_count_within_3km",
        "cafe_restaurant_count_within_3km"
    ]
    df["total_services_count_3km"] = df[count_cols].sum(axis=1)

    # Distances
    dist_cols = [
        "dist_nearest_school_km",
        "dist_nearest_hospital_km",
        "dist_nearest_supermarket_km",
        "dist_nearest_mall_km",
        "dist_nearest_transit_station_km",
        "dist_nearest_cafe_restaurant_km"
    ]

    df["avg_distance_services_3km"] = df[dist_cols].mean(axis=1)
    df["min_distance_services_3km"] = df[dist_cols].min(axis=1)

    # Accessibility
    df["accessibility_score"] = sum(
        1 / (df[col] + 1) for col in dist_cols
    )

    return df

In [ ]:
X_train = add_arithmetic_features(X_train)
X_val   = add_arithmetic_features(X_val)
X_test  = add_arithmetic_features(X_test)

# aggregation

In [ ]:
def fit_location_stats(df):
    df = df.copy()

    # District stats on train
    district_stats = df.groupby("district").agg({
        "area_value": "mean",
        "bedrooms": "mean",
        "bathroom": "mean",
        "total_services_count_3km": "mean"
    }).rename(columns={
        "area_value": "district_avg_area",
        "bedrooms": "district_avg_bedrooms",
        "bathroom": "district_avg_bathroom",
        "total_services_count_3km": "district_avg_services"
    })

    # Town stats on train
    town_stats = df.groupby("town").agg({
        "area_value": "mean",
        "bedrooms": "mean",
        "bathroom": "mean"
    }).rename(columns={
        "area_value": "town_avg_area",
        "bedrooms": "town_avg_bedrooms",
        "bathroom": "town_avg_bathroom"
    })

    return district_stats, town_stats

In [ ]:
def apply_location_stats(df, district_stats, town_stats):
    df = df.copy()

    # --- Merge precomputed stats ---
    df = df.merge(district_stats, on="district", how="left")
    df = df.merge(town_stats, on="town", how="left")

    # --- Relative features ---
    df["area_vs_district_avg"] = (df["area_value"] - df["district_avg_area"]) / df["district_avg_area"]
    df["bedrooms_vs_district_avg"] = (df["bedrooms"] - df["district_avg_bedrooms"]) / df["district_avg_bedrooms"]
    df["bathroom_vs_district_avg"] = (df["bathroom"] - df["district_avg_bathroom"]) / df["district_avg_bathroom"]

    return df

In [ ]:
# get stats from train only
district_stats, town_stats = fit_location_stats(X_train)

# Apply on every split
X_train = apply_location_stats(X_train, district_stats, town_stats)
X_val   = apply_location_stats(X_val, district_stats, town_stats)
X_test  = apply_location_stats(X_test, district_stats, town_stats)

# boolean features

In [ ]:
def apply_binary_features(df, area_median):
    df = df.copy()

    df["is_large_house"] = df["area_value"] > area_median
    df["is_small_house"] = df["area_value"] < area_median

    df["near_school"] = df["dist_nearest_school_km"] < 1
    df["near_mall"] = df["dist_nearest_mall_km"] < 2

    df["high_quality_listing"] = (df["is_premium"] == 1) | (df["is_featured"] == 1)
    

    return df

In [ ]:
area_median = X_train["area_value"].median()

X_train = apply_binary_features(X_train, area_median)
X_val   = apply_binary_features(X_val, area_median)
X_test  = apply_binary_features(X_test, area_median)

In [ ]:
# print(len(X_train.columns))

# Feature Selection
-   Filter Methods
-   Wrapper Methods
-   pemutation importance

### Step 1: Variance Threshold

In [ ]:
selector = VarianceThreshold(threshold=0.1)

X_train_var = selector.fit_transform(X_train)

selected_features = X_train.columns[selector.get_support()]

X_train = pd.DataFrame(X_train_var, columns=selected_features, index=X_train.index)
X_val = X_val[selected_features]
X_test = X_test[selected_features]

print("Remaining features:", len(selected_features))
print(selected_features)

log_transformation_action(  
    stage="Feature Selection",
    column=len(selected_features),
    action="Variance Thresholding",
    reason="To remove features that have very low variance, which are unlikely to contribute to model performance."     
)

### Step 2: Correlation with target

In [ ]:
corr = X_train.corrwith(y_train).abs().sort_values(ascending=False)
corr.head(20).plot(kind="barh")
plt.title("Top correlated features with target")
plt.show()


**Remove weak target correlation**

In [ ]:
selected_corr = corr[corr > 0.005].index.tolist()
print(len(selected_corr))

X_train = X_train[selected_corr]
X_val = X_val[selected_corr]
X_test = X_test[selected_corr]

log_transformation_action(
    stage="Feature Selection",
    column=len(selected_corr),
    action="Correlation Thresholding",
    reason="To remove features that have a very low correlation with the target variable, which are unlikely to contribute to model performance."     
)


### Step 3: Remove multicollinearity

In [ ]:
# Find and remove multicollinear features
def find_correlated_features(corr_matrix, threshold=0.9):
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    correlated_pairs = []
    for column in upper.columns:
        if column == 'target':
            continue
        corr_feats = upper. index[abs(upper[column]) > threshold].tolist()
        for cf in corr_feats:
            if cf != 'target':
                correlated_pairs. append({
                'feature_A': column,
                'feature_B': cf,
                'correlation': round(corr_matrix. loc[column, cf], 3),
                })

    return pd. DataFrame(correlated_pairs)

In [ ]:
df_corr=X_train.copy()
df_corr["target"] = y_train
corr_matrix = df_corr.corr()
print(corr_matrix)
print("\nCorrelation with target (sorted):")
print(corr_matrix['target'].sort_values(ascending=False).round(4).to_string())

In [ ]:
correlated_features = find_correlated_features(corr_matrix, threshold=0.9)
print("\nHighly correlated feature pairs (|r| > 0.9):")
print(correlated_features.to_string(index=False))


features_to_remove = set()
for _, row in correlated_features.iterrows():
    feat1, feat2 = row['feature_A'], row['feature_B']
    corr_1_t = abs(corr_matrix.loc[feat1, 'target'])
    corr_2_t = abs(corr_matrix.loc[feat2, 'target'])
    features_to_remove.add(feat2 if corr_1_t > corr_2_t else feat1)

features_to_keep = [
    c for c in df_corr.columns if c not in features_to_remove and c != 'target'
]

In [ ]:
print(features_to_remove)
print(features_to_keep)

In [ ]:
len(features_to_keep)

In [ ]:


X_train = X_train[features_to_keep]
X_val = X_val[features_to_keep]
X_test = X_test[features_to_keep]

log_transformation_action(
    stage="Feature Selection",
    column=len(features_to_keep),
    action="Correlation Analysis",
    reason="To remove highly correlated features while retaining the one with the strongest correlation to the target variable, thus reducing multicollinearity and improving model performance."     
)

In [ ]:


plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap")
plt.show()


## 2. Wrapper Method


In [ ]:

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)



### RFECV

In [ ]:
from sklearn.feature_selection import RFECV

rfecv = RFECV(
    estimator=model,
    step=1,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rfecv.fit(X_train, y_train)

In [ ]:
selected_rfecv = X_train.columns[rfecv.support_]

print("Optimal features:", len(selected_rfecv))
print(selected_rfecv)

In [ ]:
X_train = X_train[selected_rfecv]
X_val = X_val[selected_rfecv]
X_test = X_test[selected_rfecv]

log_transformation_action(
    stage="Feature Selection",
    column=len(selected_rfecv),
    action="RFECV",
    reason="To automatically select the optimal number of features based on cross-validated model performance, while eliminating less important features."     
)

### 

In [ ]:
# save transformation log report
transformation_log_df = pd.DataFrame(transformation_log_Report) 
transformation_log_df.to_csv(TRANSFORMATION_LOG_REPORT_PATH, index=False)

# save transformed datasets
X_train.to_csv("./../data/transformed/X_train.csv", index=False)
X_val.to_csv("./../data/transformed/X_val.csv", index=False)
X_test.to_csv("./../data/transformed/X_test.csv", index=False)
